In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 수동 가중치 설정
manual_weights = {
    '항공운송': {'normalized_score': 0.6, 'score_ma3': 0.2, 'volatility': 0.1, 'score_change': 0.1},
    '전자부품': {'normalized_score': 0.4, 'industry_avg_foreign_vol': 0.3, 'industry_avg_volume': 0.3},
    '바이오': {'score_ma5': 0.5, 'normalized_score': 0.3, 'volatility': 0.1, 'score_change': 0.1},
    '에너지': {'score_ma3': 0.5, 'normalized_score': 0.3, 'volatility': 0.2},
    '조선업': {'score_change': 0.4, 'normalized_score': 0.4, 'score_ma3': 0.2},
    '금속제조업': {'score_ma3': 0.3, 'score_change': 0.3, 'volatility': 0.2, 'normalized_score': 0.2},
    '반도체제조업': {'foreign_ratio': 0.3, 'score_change': 0.2, 'volatility': 0.2, 'score_ma3': 0.3},
    '자동차': {'normalized_score': 0.2, 'score_change': 0.6, 'score_ma3': 0.2},
    '정보통신업': {'normalized_score': 0.1, 'score_ma5': 0.8, 'score_change': 0.1},
    '건설업': {'normalized_score': 0.1, 'volatility': 0.1, 'score_ma5': 0.8},
    '석유정제': {'normalized_score': 0.2, 'score_ma5': 0.8},
    '금융업': {'normalized_score': 0.5, 'volume': 0.2, 'institution_vol': 0.15, 'foreign_vol': 0.15},
    '보험업':  {'normalized_score': 0.3,'volume': 0.25, 'institution_vol': 0.2, 'foreign_ratio': 0.15, 'foreign_vol': 0.15},
    '식료품': {'normalized_score': 0.2, 'volume': 0.2, 'institution_vol': 0.1, 'foreign_vol': 0.1, 'foreign_ratio': 0.1, 'volatility': 0.1,'score_change': 0.1, 'score_ma3': 0.1},
    '방송업':  {'normalized_score': 0.6, 'foreign_vol': 0.15, 'institution_vol': 0.15, 'score_change': 0.1}
}

# 모델 선택 함수 (CatBoost는 class weight 반영)
def get_model(industry, y_train=None):
    if industry in ['자동차', '정보통신업', '건설업', '석유정제']:
        if y_train is not None:
            pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
            return CatBoostClassifier(random_seed=42, verbose=0, class_weights=[1, pos_weight])
        else:
            return CatBoostClassifier(random_seed=42, verbose=0)
    else:
        return LGBMClassifier(random_state=42, is_unbalance=True)

# 시프트 설정
def get_t_shift(industry):
    return {'전자부품': 3, '정보통신업': 3}.get(industry, 1)

# 훈련 및 평가
def train_all_industries(df):
    df = df.copy()
    df['adjusted_date'] = pd.to_datetime(df['adjusted_date'], dayfirst=True, errors='coerce')
    df = df.sort_values(by=['industry', 'adjusted_date'])

    # 파생 피처 생성
    df['industry_avg_close'] = df.groupby(['industry', 'adjusted_date'])['close'].transform('mean')
    df['industry_avg_foreign_vol'] = df.groupby(['industry', 'adjusted_date'])['foreign_vol'].transform('mean')
    df['industry_avg_volume'] = df.groupby(['industry', 'adjusted_date'])['volume'].transform('mean')
    df['score_change'] = df.groupby('industry')['normalized_score'].diff()
    df['score_ma3'] = df.groupby('industry')['normalized_score'].rolling(3).mean().reset_index(0, drop=True)
    df['score_ma5'] = df.groupby('industry')['normalized_score'].rolling(5).mean().reset_index(0, drop=True)
    df['volatility'] = df.groupby('industry')['industry_avg_close'].rolling(3).std().reset_index(0, drop=True)

    result_rows = []

    for industry, weights in manual_weights.items():
        print(f"\n[산업군: {industry}]")
        sub = df[df['industry'] == industry].copy().sort_values('adjusted_date')

        t_shift = get_t_shift(industry)
        sub['future_close'] = sub['industry_avg_close'].shift(-t_shift)
        sub['log_return'] = np.log(sub['future_close'] / sub['industry_avg_close'])
        sub['label'] = (sub['log_return'] > 0.01).astype(int)

        features = list(weights.keys())
        sub = sub.dropna(subset=features + ['label'])
        if sub.empty:
            print("No data available after dropping NA.")
            continue

        sub['manual_weighted_score'] = sum(sub[feat] * w for feat, w in weights.items())

        split_point = sub['adjusted_date'].dt.year >= 2025
        X_train, X_test = sub[~split_point][features], sub[split_point][features]
        y_train, y_test = sub[~split_point]['label'], sub[split_point]['label']

        model = get_model(industry, y_train)
        model.fit(X_train, y_train)

        try:
            proba = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, proba)
        except:
            auc = None
            proba = np.zeros(len(X_test))

        preds = (proba > 0.3).astype(int)  # threshold 조정
        acc = accuracy_score(y_test, preds)
        cm = confusion_matrix(y_test, preds)

        print(f"Accuracy (T+{t_shift}): {acc:.4f}")
        if auc is not None:
            print(f"AUC: {auc:.4f}")
        print("Confusion Matrix:")
        print(cm)

        result_rows.append({
            'industry': industry,
            'model': model,
            'features': features,
            't_shift': t_shift,
            'accuracy': acc,
            'auc': auc,
            'confusion_matrix': cm.tolist(),
            'manual_weights': weights
        })

    return pd.DataFrame(result_rows)

# 실행 예시
df = pd.read_csv('v3_news_fin_full_merge.csv')
results = train_all_industries(df)
results

In [ ]:
# 수익률 비교 그래프 (마켓 vs 전략)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

manual_weights = {
    '항공운송': {'normalized_score': 0.7, 'score_ma3': 0.1, 'volatility': 0.1, 'score_change': 0.1},
    '전자부품': {'normalized_score': 0.4, 'industry_avg_foreign_vol': 0.3, 'industry_avg_volume': 0.3},
    '바이오': {'score_ma5': 0.5, 'normalized_score': 0.3, 'volatility': 0.1, 'score_change': 0.1},
    '에너지': {'score_ma3': 0.5, 'normalized_score': 0.3, 'volatility': 0.2},
    '조선업': {'score_change': 0.4, 'normalized_score': 0.4, 'score_ma3': 0.2},
    '금속제조업': {'score_ma3': 0.3, 'score_change': 0.3, 'volatility': 0.2, 'normalized_score': 0.2},
    '반도체제조업': {'foreign_ratio': 0.3, 'score_change': 0.2, 'volatility': 0.2, 'score_ma3': 0.3},
    '자동차': {'normalized_score': 0.2, 'score_change': 0.6, 'score_ma3': 0.2},
    '정보통신업': {'normalized_score': 0.1, 'score_ma5': 0.8, 'score_change': 0.1},
    '건설업': {'normalized_score': 0.1, 'volatility': 0.1, 'score_ma5': 0.8},
    '석유정제': {'normalized_score': 0.2, 'score_ma5': 0.8},
    '금융업': {'normalized_score': 0.5, 'volume': 0.2, 'institution_vol': 0.15, 'foreign_vol': 0.15},
    '보험업': {'normalized_score': 0.3,'volume': 0.25, 'institution_vol': 0.2, 'foreign_ratio': 0.15, 'foreign_vol': 0.15},
    '식료품': {'normalized_score': 0.2, 'volume': 0.2, 'institution_vol': 0.1, 'foreign_vol': 0.1, 'foreign_ratio': 0.1, 'volatility': 0.1,'score_change': 0.1, 'score_ma3': 0.1},
    '방송업': {'normalized_score': 0.7,  'score_change': 0.3}
}

#색상 매핑
industry_colors = {
    '항공운송': '#1f77b4', '전자부품': '#ff7f0e', '바이오': '#2ca02c', '에너지': '#d62728',
    '조선업': '#9467bd', '금속제조업': '#8c564b', '반도체제조업': '#e377c2', '자동차': '#7f7f7f',
    '정보통신업': '#bcbd22', '건설업': '#17becf', '석유정제': '#aec7e8', '금융업': '#ffbb78',
    '보험업': '#98df8a', '식료품': '#ff9896', '방송업': '#c5b0d5'
}

# 산업군별 모델
def get_model(industry, y_train=None):
    if industry in ['자동차', '정보통신업', '건설업', '석유정제', '금융업']:
        pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        return CatBoostClassifier(random_seed=42, verbose=0, class_weights=[1, pos_weight])
    else:
        return LGBMClassifier(random_state=42, is_unbalance=True)

#시차 설정
def get_t_shift(industry):
    return {'전자부품': 3, '정보통신업': 3}.get(industry, 1)

def plot_strategy_vs_hold_upgraded(df, optimal_buy_data):
    df['adjusted_date'] = pd.to_datetime(df['adjusted_date'], dayfirst=True, errors='coerce')
    df = df.sort_values(by=['industry', 'adjusted_date'])

    tbuy_map = dict(zip(optimal_buy_data['industry'], optimal_buy_data['T_buy']))

    # 파생 피처 생성
    df['industry_avg_close'] = df.groupby(['industry', 'adjusted_date'])['close'].transform('mean')
    df['industry_avg_foreign_vol'] = df.groupby(['industry', 'adjusted_date'])['foreign_vol'].transform('mean')
    df['industry_avg_volume'] = df.groupby(['industry', 'adjusted_date'])['volume'].transform('mean')
    df['score_change'] = df.groupby('industry')['normalized_score'].diff()
    df['score_ma3'] = df.groupby('industry')['normalized_score'].rolling(3).mean().reset_index(0, drop=True)
    df['score_ma5'] = df.groupby('industry')['normalized_score'].rolling(5).mean().reset_index(0, drop=True)
    df['volatility'] = df.groupby('industry')['industry_avg_close'].rolling(3).std().reset_index(0, drop=True)

    industries = list(manual_weights.keys())
    n_rows, n_cols = 3, 5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(26, 16), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, industry in enumerate(industries):
        ax = axes[i]
        weights = manual_weights[industry]
        sub = df[df['industry'] == industry].copy().dropna(subset=list(weights.keys()))
        if sub.empty or industry not in tbuy_map:
            ax.set_visible(False)
            continue

        t_shift = get_t_shift(industry)
        sub['future_close'] = sub['industry_avg_close'].shift(-t_shift)
        sub['log_return'] = np.log(sub['future_close'] / sub['industry_avg_close'])

        sub['manual_weighted_score'] = sum(sub[feat] * w for feat, w in weights.items())
        sub = sub.dropna(subset=['manual_weighted_score', 'log_return'])

        train = sub[sub['adjusted_date'].dt.year < 2025].copy()
        test = sub[sub['adjusted_date'].dt.year == 2025].copy()
        if train.empty or test.empty:
            ax.set_visible(False)
            continue

        X_train = train[list(weights.keys())]
        y_train = (train['log_return'] > 0.01).astype(int)
        X_test = test[list(weights.keys())]
        y_test = (test['log_return'] > 0.01).astype(int)

        model = get_model(industry, y_train)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        T_buy = tbuy_map[industry]
        test['pred'] = (proba > T_buy).astype(int)

        test['strategy_return'] = test['log_return'] * test['pred']
        test['cum_strategy_return'] = test['strategy_return'].cumsum()
        test['cum_hold_return'] = test['log_return'].cumsum()

        # 산업군별 전략 색 / 보유는 고정 회색
        strategy_color = industry_colors.get(industry, 'blue')
        ax.plot(test['adjusted_date'], test['cum_strategy_return'], label='전략',
                color=strategy_color, linewidth=2.5)
        ax.plot(test['adjusted_date'], test['cum_hold_return'], label='보유',
                color='dimgray', linestyle='--', linewidth=1.8, alpha=0.8)

        ax.set_title(f"{industry} (T_buy={T_buy})", fontsize=14)
        ax.tick_params(labelsize=11)
        ax.axhline(0, linestyle='--', color='black', alpha=0.3)
        ax.grid(True)

        # 날짜 포맷 간소화
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

        # 범례는 첫 번째 subplot에만
        if i == 0:
            ax.legend(fontsize=11)

    for j in range(len(industries), n_rows * n_cols):
        axes[j].set_visible(False)

    fig.suptitle("산업군별 전략 vs 보유 누적 수익률 (2025)", fontsize=20)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

df = pd.read_csv("v3_news_fin_full_merge.csv")

optimal_buy_data = {
    "industry": [
        "건설업", "금속제조업", "금융업", "바이오", "반도체제조업", "방송업", "보험업",
        "석유정제", "식료품", "에너지", "자동차", "전자부품", "정보통신업", "조선업", "항공운송"
    ],
    "T_buy": [
        0.1, 0.7, 0.4, 0.5, 0.2, 0.1, 0.2, 0.4, 0.1, 0.4, 0.6, 0.2, 0.1, 0.2, 0.4
    ]
}

df = pd.read_csv('v3_news_2020_2025_06_combined.csv')

plot_strategy_vs_hold_upgraded(df, optimal_buy_data)

In [ ]:
industry_colors = {
    '건설업': 'blue',
    '금속제조업': 'olive',
    '금융업': 'cyan',
    '바이오': 'orange',
    '반도체제조업': 'coral',
    '방송업': 'lime',
    '보험업': 'gold',
    '석유정제': 'magenta',
    '식료품': 'brown',
    '에너지': 'red',
    '자동차': 'navy',
    '전자부품': 'green',
    '정보통신업': 'teal',
    '조선업': 'purple',
    '항공운송': 'skyblue'
}

def plot_strategy_per_industry(df, optimal_buy_data):
    df['adjusted_date'] = pd.to_datetime(df['adjusted_date'], dayfirst=True, errors='coerce')
    df = df.sort_values(by=['industry', 'adjusted_date'])

    tbuy_map = dict(zip(optimal_buy_data['industry'], optimal_buy_data['T_buy']))

    # 파생 피처 생성
    df['industry_avg_close'] = df.groupby(['industry', 'adjusted_date'])['close'].transform('mean')
    df['industry_avg_foreign_vol'] = df.groupby(['industry', 'adjusted_date'])['foreign_vol'].transform('mean')
    df['industry_avg_volume'] = df.groupby(['industry', 'adjusted_date'])['volume'].transform('mean')
    df['score_change'] = df.groupby('industry')['normalized_score'].diff()
    df['score_ma3'] = df.groupby('industry')['normalized_score'].rolling(3).mean().reset_index(0, drop=True)
    df['score_ma5'] = df.groupby('industry')['normalized_score'].rolling(5).mean().reset_index(0, drop=True)
    df['volatility'] = df.groupby('industry')['industry_avg_close'].rolling(3).std().reset_index(0, drop=True)

    summary_data = []

    for industry in manual_weights.keys():
        if industry not in tbuy_map:
            continue

        weights = manual_weights[industry]
        sub = df[df['industry'] == industry].copy().dropna(subset=list(weights.keys()))
        if sub.empty:
            continue

        t_shift = get_t_shift(industry)
        sub['future_close'] = sub['industry_avg_close'].shift(-t_shift)
        sub['log_return'] = np.log(sub['future_close'] / sub['industry_avg_close'])

        sub['manual_weighted_score'] = sum(sub[feat] * w for feat, w in weights.items())
        sub = sub.dropna(subset=['manual_weighted_score', 'log_return'])

        train = sub[sub['adjusted_date'].dt.year < 2025].copy()
        test = sub[sub['adjusted_date'].dt.year == 2025].copy()

        if train.empty or test.empty:
            continue

        X_train = train[list(weights.keys())]
        y_train = (train['log_return'] > 0.01).astype(int)
        X_test = test[list(weights.keys())]

        model = get_model(industry, y_train)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        T_buy = tbuy_map[industry]
        test['pred'] = (proba > T_buy).astype(int)  #구매

        test['strategy_return'] = test['log_return'] * test['pred']
        test['cum_strategy_return'] = test['strategy_return'].cumsum()
        test['cum_hold_return'] = test['log_return'].cumsum()

        # 초과 수익률 계산
        excess_return = test['cum_strategy_return'].iloc[-1] - test['cum_hold_return'].iloc[-1]
        daily_excess_mean = (test['strategy_return'] - test['log_return']).mean()
        summary_data.append({
            'industry': industry,
            't_shift': t_shift,
            'T_buy': T_buy,
            'excess_return': excess_return,
            'daily_excess_mean': daily_excess_mean
        })

        # 그래프
        plt.figure(figsize=(8, 4))
        strategy_color = industry_colors.get(industry, 'blue')
        plt.plot(test['adjusted_date'], test['cum_strategy_return'], label='전략', color=strategy_color)
        plt.plot(test['adjusted_date'], test['cum_hold_return'], label='보유', color='dimgray', linestyle='--', alpha=0.8)
        plt.title(f"{industry} : 보유vs전략 (T_buy={T_buy}) 6개월(2025.01~06)", fontsize=14)
        plt.axhline(0, linestyle='--', color='black', alpha=0.2)
        plt.xlabel("날짜")
        plt.ylabel("누적 수익률")
        plt.grid(True)
        plt.legend()
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(f"6month/{industry}_strategy_6month.png")
        plt.close()

    # 결과 요약 테이블 출력
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values(by='excess_return', ascending=False)
    print(summary_df[['industry', 'excess_return', 'daily_excess_mean']])

    # 최대, 평균 초과 수익률
    max_row = summary_df.iloc[0]
    avg_excess = summary_df['excess_return'].mean()
    print(f"\n최대 초과 수익률: +{max_row['excess_return']:.2%} (T+{max_row['t_shift']}, {max_row['industry']})")
    print(f"평균 초과 수익률: +{avg_excess:.2%}")
